# Urdu Question Generation — training run

A **from-scratch 2-layer BiLSTM encoder–decoder with Bahdanau attention**. Given an Urdu
sentence with the answer wrapped in `<ans> … </ans>`, the model generates the question that
span answers. No pretrained weights, no Transformers — only `nn.Embedding`, `nn.LSTM`,
`nn.Linear`.

Every step below calls a module in the repo's `src/` package, so the code is identical here
and locally, and each step can be pointed to directly in the viva.

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (P100 is compute-capability 6.0; Kaggle's PyTorch has no kernels for it. T4 is 7.5. The code uses one T4.) |
| Internet | **On** (needed to download the dataset) |

Pipeline: setup → data prep → tokenizer → debug gate → full training (~75–90 min) → evaluation → bundle outputs.

## 1. Setup

Clone the repo, install the three libraries the Kaggle image is missing
(`sentencepiece`, `sacrebleu`, `rouge-score`), and confirm the GPU can actually run CUDA
kernels — `CUDA kernels OK: [2.0, 2.0, 2.0]` means we are good to train.

In [ ]:
%cd /kaggle/working
!rm -rf urdu-question-generation
!git clone --depth 1 https://github.com/Hanzala-12/urdu-question-generation.git
%cd urdu-question-generation
!pip -q install sentencepiece sacrebleu rouge-score arabic-reshaper python-bidi
!apt-get -qq install -y fonts-noto-core > /dev/null 2>&1 || true
import torch
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU!')
print('CUDA kernels OK:', (torch.ones(3, device='cuda') * 2).tolist())

## 2. Task 1 — Data preparation

Load **UQA** (train + validation) and **Wiki-UQA** (out-of-domain test). For every
answerable row: find the *sentence* that contains the answer using its character offset,
verify the offset, wrap the span in `<ans> … </ans>`, and drop pairs longer than 60
(source) / 25 (target) whitespace tokens.

Why sentence-level, not the whole paragraph: a from-scratch RNN on ~75k pairs cannot learn
to read a 300-token context; restricting to the answer sentence (Du et al., 2017) is what
makes the task learnable. Writes `data/{train,valid,wiki}.tsv`, `results/data_stats.json`
(Table 1) and `results/figures/length_hist.png`.

In [ ]:
!python -m src.data_prep

## 3. Task 2 — SentencePiece tokenizer

Train a **unigram** subword model (`vocab_size=8000`, `character_coverage=1.0`) on the
training text only. `<ans>` / `</ans>` are registered as user-defined symbols so a tag is
always exactly one token; ids are pinned pad=0, unk=1, bos=2, eos=3.

Unigram (vs BPE) keeps frequent whole words and splits only rarer inflected forms into a
stem + affix pieces — a good fit for Urdu morphology. Full worked examples go to
`results/tokenizer_examples.txt`.

In [ ]:
!python -m src.spm_train

## 4. Debug gate

Train on a 10k-pair subset for a single epoch. If the loss does **not** fall, there is a
bug (bad masking, wrong target shift, wrong learning rate) — cheaper to catch in ~40 s than
after an hour. Expect validation loss to drop below ~6.

In [ ]:
!python -m src.train --debug

## 5. Task 3 — Full training

15 epochs, batch 64, Adam (lr 1e-3) with `ReduceLROnPlateau`, gradient clipping 1.0,
padding-masked cross-entropy, teacher forcing 0.5. One epoch line per epoch:
`train` and `val` cross-entropy, validation perplexity, learning rate, seconds.

Saves `artifacts/best.pt` (weights only, lowest validation loss — used by evaluation and
the front end), `results/loss_log.csv` and `results/figures/loss_curve.png`.
Runtime ~75–90 min on a T4.

In [ ]:
!python -m src.train --epochs 15 --batch-size 64

## 6. Task 4 — Evaluation

Decode UQA-validation and Wiki-UQA with **greedy** and **beam** (k=5) search and report,
per split and strategy: BLEU-4 (sacrebleu, corpus), ROUGE-L (F), perplexity
(`exp` of teacher-forced cross-entropy), and `<unk>`% of generated tokens.

Writes `results/metrics.json`, `results/samples.tsv` (≥ 50 rows: source, reference,
greedy, beam), `results/tables.md` and `results/figures/attention.png`. Sanity band:
BLEU-4 ~6–13 on UQA-valid, lower on Wiki-UQA; near 0 means a bug, above 30 means leakage.

In [ ]:
!python -m src.evaluate --split both --beam-max 3000
import json
print(json.dumps(json.load(open('results/metrics.json')), indent=2, ensure_ascii=False))

## 7. Package outputs

Copy `artifacts/` (tokenizer + `best.pt`) and `results/` (metrics, samples, tables,
figures) into one `/kaggle/working/outputs.zip` to download and unpack into the local
repo, where the front end and the write-ups use them.

In [ ]:
import shutil
shutil.copytree('artifacts', '/kaggle/working/outputs/artifacts', dirs_exist_ok=True)
shutil.copytree('results', '/kaggle/working/outputs/results', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/outputs', 'zip', '/kaggle/working/outputs')
print('done -> /kaggle/working/outputs.zip')
!ls -lhR /kaggle/working/outputs